# 01 · Identifiers

Fetches commercial establishments from OSM around Times Square, cleans and labels them.

**Outputs:**
- `csv/00_base_data.csv` — full cleaned dataset (used by all other notebooks)
- `csv/01_identifiers.csv` — `osm_id`, `lat`, `lon`

In [ ]:
import overpy
import requests
import pandas as pd
import math, time, os

os.makedirs("csv", exist_ok=True)
print(f"overpy {overpy.__version__} · pandas {pd.__version__}")

In [ ]:
# ── Parameters ─────────────────────────────────────────
LAT              = 40.7589
LON              = -73.9851
WALK_MINUTES     = 15.0
WALK_SPEED_M_MIN = 80.0
RADIUS_M         = WALK_MINUTES * WALK_SPEED_M_MIN

# ── OSM tag config ────────────────────────────────────
COMMERCIAL_TAGS = ["shop", "amenity", "office", "craft", "tourism", "leisure"]

AMENITY_EXCLUDE = {
    "parking", "bench", "waste_basket", "recycling",
    "post_box", "bicycle_parking", "drinking_water",
    "telephone", "toilets", "street_lamp", "fire_station",
    "school", "place_of_worship", "clock",
    "bicycle_rental", "parking_entrance", "vending_machine",
    "fountain", "charging_station", "device_charging_station",
    "bus_station", "social_facility", "community_centre",
    "police", "university", "college", "kindergarten",
}
LEISURE_EXCLUDE = {
    "garden", "park", "pitch", "playground", "outdoor_seating",
    "swimming_pool", "dog_park", "nature_reserve", "picnic_site",
    "shelter", "watering_place", "common",
}
TOURISM_EXCLUDE = {
    "artwork", "viewpoint", "information", "attraction",
    "picnic_site", "camp_site", "caravan_site",
}
SHOP_EXCLUDE = {"vacant", "yes"}

OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]
HEADERS = {"User-Agent": "osmnx-data-scraper/1.0 (research project)"}

print(f"Origin: ({LAT}, {LON}) · Radius: {RADIUS_M:.0f} m")

In [ ]:
def build_overpass_query(lat, lon, radius):
    tag_filters = "\n    ".join(
        [f'node["{tag}"](around:{radius:.0f},{lat},{lon});' for tag in COMMERCIAL_TAGS]
        + [f'way["{tag}"](around:{radius:.0f},{lat},{lon});' for tag in COMMERCIAL_TAGS]
    )
    return f"""[out:json][timeout:60];\n(\n    {tag_filters}\n);\nout center tags;"""


def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = math.radians(lat2 - lat1), math.radians(lon2 - lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(a))


def _extract_record(osm_id, osm_type, lat, lon, tags, origin_lat, origin_lon):
    primary_tag = primary_value = None
    for tag in COMMERCIAL_TAGS:
        if tag in tags:
            val = tags[tag]
            if tag == "shop"    and val in SHOP_EXCLUDE:    return None
            if tag == "amenity" and val in AMENITY_EXCLUDE: return None
            if tag == "leisure" and val in LEISURE_EXCLUDE: return None
            if tag == "tourism" and val in TOURISM_EXCLUDE: return None
            primary_tag, primary_value = tag, val
            break
    if not primary_tag:
        return None
    return {
        "osm_id": osm_id, "osm_type": osm_type,
        "lat": round(lat, 7), "lon": round(lon, 7),
        "primary_tag": primary_tag, "primary_value": primary_value,
        "name": tags.get("name", ""),
    }


def fetch_osm_data(lat, lon, radius):
    query = build_overpass_query(lat, lon, radius)
    print(f"  Querying Overpass API (radius: {radius:.0f} m)...")
    last_error = None
    for ep in OVERPASS_ENDPOINTS:
        try:
            r = requests.post(ep, data={"data": query}, headers=HEADERS, timeout=90)
            r.raise_for_status()
            result = overpy.Overpass().parse_json(r.text)
            print(f"  OK via {ep}")
            break
        except Exception as e:
            last_error = e
            print(f"  x {ep}: {e}")
            time.sleep(3)
    else:
        raise RuntimeError(f"All endpoints failed: {last_error}")

    records = []
    for node in result.nodes:
        rec = _extract_record(node.id, "node", float(node.lat), float(node.lon),
                              node.tags, lat, lon)
        if rec: records.append(rec)
    for way in result.ways:
        if hasattr(way, "center_lat") and way.center_lat:
            rec = _extract_record(way.id, "way", float(way.center_lat), float(way.center_lon),
                                  way.tags, lat, lon)
            if rec: records.append(rec)
    print(f"  {len(records)} commercial features found")
    return records

print("Functions ready.")

In [ ]:
records = fetch_osm_data(LAT, LON, RADIUS_M)
if not records:
    raise ValueError("No commercial features found.")
df = pd.DataFrame(records)
print(f"{len(df)} raw records")

In [ ]:
# ── Label taxonomy ────────────────────────────────────
DROP_VALUES = {
    "outdoor_seating", "pitch", "playground", "shelter",
    "watering_place", "picnic_site", "dog_park", "nature_reserve",
    "common", "artwork", "viewpoint", "information", "attraction",
    "sightseeing", "guide", "loading_dock", "outpost", "post_depot",
    "diplomatic", "yes", "library", "educational_institution",
    "prep_school", "parish", "rectory", "charity",
    "post_office", "car", "bicycle", "hairdresser_supply",
    "newspaper", "vacuum_cleaner", "video", "sauce",
    "watchmaker", "cleaning", "locksmith", "magic",
    "conference_centre", "fuel", "trade", "training",
    "car_repair", "funeral_directors",
}

LABEL_MAP = {
    "restaurant": "food_drink", "fast_food": "food_drink",
    "cafe": "food_drink", "bar": "food_drink",
    "pub": "food_drink", "food_court": "food_drink",
    "bakery": "food_drink", "ice_cream": "food_drink",
    "deli": "food_drink", "confectionery": "food_drink",
    "chocolate": "food_drink", "coffee": "food_drink",
    "seafood": "food_drink", "pastry": "food_drink",
    "tea": "food_drink", "beverages": "food_drink",
    "health_food": "food_drink", "brewery": "food_drink",
    "salad": "food_drink", "butcher": "food_drink",
    "grocery": "food_drink", "kiosk": "food_drink",
    "greengrocer": "food_drink",
    "clothes": "retail", "gift": "retail",
    "jewelry": "retail", "shoes": "retail",
    "convenience": "retail", "supermarket": "retail",
    "department_store": "retail", "mobile_phone": "retail",
    "newsagent": "retail", "alcohol": "retail",
    "watches": "retail", "cosmetics": "retail",
    "toys": "retail", "electronics": "retail",
    "stationery": "retail", "bag": "retail",
    "wine": "retail", "books": "retail",
    "fabric": "retail", "fashion_accessories": "retail",
    "nutrition_supplements": "retail", "houseware": "retail",
    "furniture": "retail", "art": "retail",
    "antiques": "retail", "video_games": "retail",
    "musical_instrument": "retail", "vinyl": "retail",
    "collector": "retail", "camera": "retail",
    "hifi": "retail", "hardware": "retail",
    "outdoor": "retail", "pet": "retail",
    "perfumery": "retail", "chemist": "retail",
    "variety_store": "retail", "mall": "retail",
    "tobacco": "retail", "cannabis": "retail",
    "e-cigarette": "retail", "florist": "retail",
    "leather": "retail", "haberdashery": "retail",
    "paint": "retail", "frame": "retail",
    "bed": "retail", "military_surplus": "retail",
    "wholesale": "retail", "erotic": "retail",
    "ticket": "retail", "marketplace": "retail",
    "craft": "retail",
    "hairdresser": "personal_care", "beauty": "personal_care",
    "massage": "personal_care", "tattoo": "personal_care",
    "shoemaker": "personal_care", "shoe_repair": "personal_care",
    "laundry": "personal_care", "dry_cleaning": "personal_care",
    "copyshop": "personal_care", "photo": "personal_care",
    "photographer": "personal_care", "sauna": "personal_care",
    "tailor": "personal_care", "sewing": "personal_care",
    "pharmacy": "health", "dentist": "health",
    "clinic": "health", "doctors": "health",
    "optician": "health", "veterinary": "health",
    "hospital": "health", "medical_supply": "health",
    "psychiatrist": "health", "physician": "health",
    "bank": "finance", "atm": "finance",
    "bureau_de_change": "finance", "financial_advisor": "finance",
    "financial": "finance", "money_lender": "finance",
    "money_transfer": "finance", "pawnbroker": "finance",
    "payment_centre": "finance",
    "hotel": "hospitality", "hostel": "hospitality", "motel": "hospitality",
    "company": "office_professional", "lawyer": "office_professional",
    "accountant": "office_professional", "it": "office_professional",
    "ngo": "office_professional", "association": "office_professional",
    "advertising_agency": "office_professional", "consulting": "office_professional",
    "engineer": "office_professional", "telecommunication": "office_professional",
    "estate_agent": "office_professional", "travel_agency": "office_professional",
    "architect": "office_professional", "tax_advisor": "office_professional",
    "marketing": "office_professional", "property_management": "office_professional",
    "union": "office_professional", "foundation": "office_professional",
    "coworking": "office_professional", "coworking_space": "office_professional",
    "theatre": "leisure_entertainment", "cinema": "leisure_entertainment",
    "fitness_centre": "leisure_entertainment", "sports": "leisure_entertainment",
    "sports_centre": "leisure_entertainment", "nightclub": "leisure_entertainment",
    "stripclub": "leisure_entertainment", "arts_centre": "leisure_entertainment",
    "gallery": "leisure_entertainment", "museum": "leisure_entertainment",
    "events_venue": "leisure_entertainment", "escape_game": "leisure_entertainment",
    "karaoke": "leisure_entertainment", "karaoke_box": "leisure_entertainment",
    "swimming_pool": "leisure_entertainment", "ice_rink": "leisure_entertainment",
    "miniature_golf": "leisure_entertainment", "dojo": "leisure_entertainment",
    "dance": "leisure_entertainment", "hackerspace": "leisure_entertainment",
    "social_centre": "leisure_entertainment", "fortune_teller": "leisure_entertainment",
    "car_rental": "leisure_entertainment", "taxi": "leisure_entertainment",
    "luggage_storage": "leisure_entertainment"
}

df_clean = df[~df["primary_value"].isin(DROP_VALUES)].copy()
df_clean["label"] = df_clean["primary_value"].map(LABEL_MAP)
df_clean = df_clean.dropna(subset=["label"]).copy()
df_clean = df_clean.reset_index(drop=True)

print(f"Cleaned: {len(df_clean)} records")
print(df_clean["label"].value_counts())

In [ ]:
# Save full base data for other notebooks
df_clean.to_csv("csv/00_base_data.csv", index=False, encoding="utf-8")

# Save identifiers only
df_clean[["osm_id", "lat", "lon"]].to_csv("csv/01_identifiers.csv", index=False, encoding="utf-8")

print(f"Saved {len(df_clean)} records")
print(f"  csv/00_base_data.csv    — full base data")
print(f"  csv/01_identifiers.csv  — osm_id, lat, lon")
df_clean[["osm_id", "lat", "lon"]].head()